# Computer Vision



**Suggested installs (run once):**
```bash
pip install numpy matplotlib opencv-python opencv-contrib-python scikit-image
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import data

rng = np.random.default_rng(42)


def show_image(img, title=None, cmap=None, figsize=(5, 5)):
    plt.figure(figsize=figsize)
    if img.ndim == 2:
        plt.imshow(img, cmap=cmap or 'gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()


def to_uint8(img):
    if img.dtype == np.uint8:
        return img
    img_min, img_max = float(img.min()), float(img.max())
    if img_max - img_min == 0:
        return np.zeros_like(img, dtype=np.uint8)
    scaled = (img - img_min) / (img_max - img_min)
    return (scaled * 255).astype(np.uint8)


## 1. Side effect of undersampling and effect of quantization of 2D analog signal

In [ ]:
size = 512
freq = 6
x = np.linspace(0, 2 * np.pi, size, endpoint=False)
X, Y = np.meshgrid(x, x)
analog_signal = np.sin(freq * X) + np.sin(freq * Y)

show_image(analog_signal, "Original 2D analog signal")


def undersample(signal, factor):
    small = signal[::factor, ::factor]
    upsampled = cv2.resize(
        small, (signal.shape[1], signal.shape[0]), interpolation=cv2.INTER_NEAREST
    )
    return small, upsampled


for factor in [2, 4, 8]:
    _, up = undersample(analog_signal, factor)
    show_image(up, f"Undersampled by {factor} (aliasing visible)")


def quantize(signal, bits):
    levels = 2 ** bits
    normalized = (signal - signal.min()) / (signal.max() - signal.min())
    quantized = np.round(normalized * (levels - 1)) / (levels - 1)
    return quantized


for bits in [8, 4, 2]:
    q = quantize(analog_signal, bits)
    show_image(q, f"Quantized to {bits}-bit")


## 2. Implement point, edge and corner detection

In [ ]:
img = data.camera()
img_u8 = to_uint8(img)

# Point detection using blob detector
params = cv2.SimpleBlobDetector_Params()
params.filterByArea = True
params.minArea = 30
params.maxArea = 5000
blob_detector = cv2.SimpleBlobDetector_create(params)
keypoints = blob_detector.detect(img_u8)
point_img = cv2.drawKeypoints(
    cv2.cvtColor(img_u8, cv2.COLOR_GRAY2BGR),
    keypoints,
    None,
    (0, 0, 255),
    cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
)

# Edge detection
edges = cv2.Canny(img_u8, 100, 200)

# Corner detection using Harris
harris = cv2.cornerHarris(np.float32(img_u8), 2, 3, 0.04)
corner_img = cv2.cvtColor(img_u8, cv2.COLOR_GRAY2BGR)
corner_img[harris > 0.01 * harris.max()] = [0, 0, 255]

show_image(point_img, "Point detection (blobs)")
show_image(edges, "Edge detection (Canny)")
show_image(corner_img, "Corner detection (Harris)")


## 3. Implement edge enhancement algorithm

In [ ]:
img = data.camera()
img_u8 = to_uint8(img)

blurred_img = cv2.GaussianBlur(img_u8, (0, 0), sigmaX=1.5)
unsharp = cv2.addWeighted(img_u8, 1.6, blurred_img, -0.6, 0)

show_image(img_u8, "Original")
show_image(unsharp, "Edge-enhanced (unsharp masking)")


## 4. Implement edge detection by Hough Transform

In [ ]:
img = data.camera()
img_u8 = to_uint8(img)

edges = cv2.Canny(img_u8, 50, 150)
lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=100, minLineLength=50, maxLineGap=10)
line_img = cv2.cvtColor(img_u8, cv2.COLOR_GRAY2BGR)

if lines is not None:
    for x1, y1, x2, y2 in lines[:, 0]:
        cv2.line(line_img, (x1, y1), (x2, y2), (0, 0, 255), 2)

show_image(edges, "Edges")
show_image(line_img, "Hough line detection")


## 5. Extract image features using SIFT algorithms

In [ ]:
color_img = data.astronaut()
img_bgr = cv2.cvtColor(color_img, cv2.COLOR_RGB2BGR)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

try:
    sift = cv2.SIFT_create()
except AttributeError as exc:
    raise RuntimeError(
        "SIFT not available. Install opencv-contrib-python to enable SIFT."
    ) from exc

keypoints, descriptors = sift.detectAndCompute(gray, None)
sift_img = cv2.drawKeypoints(
    img_bgr, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

print(f"Detected {len(keypoints)} SIFT keypoints.")
show_image(sift_img, "SIFT keypoints")


## 6. Implement simple object detection algorithm using HOG operator

In [ ]:
img = data.astronaut()
img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

(rects, weights) = hog.detectMultiScale(
    img_bgr, winStride=(8, 8), padding=(8, 8), scale=1.05
)

hog_img = img_bgr.copy()
for (x, y, w, h) in rects:
    cv2.rectangle(hog_img, (x, y), (x + w, y + h), (0, 255, 0), 2)

print(f"Detected {len(rects)} candidate objects using HOG.")
show_image(hog_img, "HOG-based detection (people detector)")


## 7. Implement RANSAC algorithms to remove outliers

In [ ]:
x = np.linspace(0, 100, 60)
y = 0.5 * x + 10 + rng.normal(0, 2, size=x.shape)

# Add outliers
outliers_x = rng.uniform(0, 100, 12)
outliers_y = rng.uniform(0, 100, 12)

points = np.column_stack([np.concatenate([x, outliers_x]), np.concatenate([y, outliers_y])])


def ransac_line(points, iterations=200, dist_threshold=2.5, min_inliers=40):
    """Fit a line with RANSAC and return the model and inlier mask.

    dist_threshold: distance threshold for classifying inliers.
    min_inliers: minimum number of inliers required for a valid model.
    returns: ((slope, intercept), inlier_mask)
    """
    best_model = None
    best_inliers = None
    best_count = 0
    n = len(points)

    for _ in range(iterations):
        idx = rng.choice(n, size=2, replace=False)
        p1, p2 = points[idx]
        if np.isclose(p1[0], p2[0]):
            continue
        slope = (p2[1] - p1[1]) / (p2[0] - p1[0])
        intercept = p1[1] - slope * p1[0]
        # Perpendicular distance from each point to the line.
        distances = np.abs(slope * points[:, 0] - points[:, 1] + intercept) / np.sqrt(
            slope**2 + 1
        )
        inliers = distances < dist_threshold
        count = inliers.sum()
        if count > best_count and count >= min_inliers:
            best_count = count
            best_model = (slope, intercept)
            best_inliers = inliers

    return best_model, best_inliers


(model, inliers) = ransac_line(points)

plt.figure(figsize=(6, 5))
plt.scatter(points[:, 0], points[:, 1], c='lightgray', label='All points')

if inliers is not None:
    plt.scatter(points[inliers, 0], points[inliers, 1], c='tab:blue', label='Inliers')
    m, b = model
    xs = np.array([0, 100])
    plt.plot(xs, m * xs + b, 'r-', label='RANSAC fit')

plt.legend()
plt.title("RANSAC line fitting with outlier removal")
plt.xlabel("x")
plt.ylabel("y")
plt.show()


## 8. Implement face detection HaarCascade classifiers

In [ ]:
img = data.astronaut()
img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(cascade_path)
faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

face_img = img_bgr.copy()
for (x, y, w, h) in faces:
    cv2.rectangle(face_img, (x, y), (x + w, y + h), (0, 255, 0), 2)

print(f"Detected {len(faces)} faces.")
show_image(face_img, "HaarCascade face detection")


## 9. Realize an algorithm for object tracking from a video

In [ ]:
def generate_synthetic_video(num_frames=40, size=(200, 200), box_size=30):
    frames = []
    h, w = size
    for i in range(num_frames):
        frame = np.zeros((h, w, 3), dtype=np.uint8)
        x = 20 + i * 3
        y = 50 + i * 2
        cv2.rectangle(frame, (x, y), (x + box_size, y + box_size), (255, 255, 255), -1)
        frames.append(frame)
    return frames


frames = generate_synthetic_video()
init_x, init_y, init_w, init_h = 20, 50, 30, 30

# Template matching tracker
template = cv2.cvtColor(
    frames[0][init_y : init_y + init_h, init_x : init_x + init_w],
    cv2.COLOR_BGR2GRAY,
)

bboxes = [(init_x, init_y, init_w, init_h)]
for frame in frames[1:]:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    res = cv2.matchTemplate(gray, template, cv2.TM_CCOEFF_NORMED)
    _, _, _, max_loc = cv2.minMaxLoc(res)
    bboxes.append((max_loc[0], max_loc[1], init_w, init_h))

last_frame = frames[-1].copy()
(x, y, w, h) = bboxes[-1]
cv2.rectangle(last_frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

show_image(frames[0], "First frame")
show_image(last_frame, "Last frame with tracked box")


## 10. IOU based bounding box construction

In [ ]:
def iou(box_a, box_b):
    ax1, ay1, aw, ah = box_a
    bx1, by1, bw, bh = box_b
    ax2, ay2 = ax1 + aw, ay1 + ah
    bx2, by2 = bx1 + bw, by1 + bh

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = aw * ah
    area_b = bw * bh
    union = area_a + area_b - inter_area
    return inter_area / union if union > 0 else 0.0


def nms(boxes, scores, iou_threshold=0.5):
    """Perform Non-Maximum Suppression and return kept indices."""
    idxs = np.argsort(scores)[::-1]
    keep = []

    while len(idxs) > 0:
        current = idxs[0]
        keep.append(current)
        remaining = []
        for idx in idxs[1:]:
            if iou(boxes[current], boxes[idx]) < iou_threshold:
                remaining.append(idx)
        idxs = np.array(remaining)

    return keep


boxes = [(30, 30, 60, 60), (40, 40, 60, 60), (120, 120, 50, 50)]
scores = [0.9, 0.75, 0.8]

selected = nms(boxes, scores, iou_threshold=0.4)
print("Selected box indices after IoU-based NMS:", selected)

canvas = np.zeros((220, 220, 3), dtype=np.uint8)
for i, (x, y, w, h) in enumerate(boxes):
    color = (0, 0, 255)
    if i in selected:
        color = (0, 255, 0)
    cv2.rectangle(canvas, (x, y), (x + w, y + h), color, 2)

show_image(canvas, "IoU-based selection (green kept)")
